# OpenHPLjl — Railway Validation and FCR/AGC Compute Notebook

Runs every validation stage on Railway and records PASS/FAIL without stopping at the first failure.


In [ ]:
import subprocess, os, time, pathlib
ROOT = pathlib.Path("/app")
os.chdir(ROOT)
results = []
def run_stage(name, cmd, timeout=1800):
    print("\n" + "="*78)
    print("STAGE:", name)
    print("CMD:", cmd)
    t0=time.time()
    p=subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    if p.stdout: print(p.stdout)
    if p.stderr:
        print("--- stderr ---")
        print(p.stderr)
    status="PASS" if p.returncode==0 else "FAIL"
    elapsed=time.time()-t0
    print(f"[{status}] {name} | exit={p.returncode} | {elapsed:.1f}s")
    results.append({"stage":name,"status":status,"exit":p.returncode,"seconds":round(elapsed,1)})
    return p.returncode


In [ ]:
run_stage('Julia version', 'julia --version', timeout=60)


In [ ]:
run_stage('Pkg.instantiate', "julia --project=. -e 'using Pkg; Pkg.instantiate()'", timeout=1800)


In [ ]:
run_stage('Pkg.test', "julia --project=. -e 'using Pkg; Pkg.test()'", timeout=1800)


In [ ]:
run_stage('Hydraulic reservoir-pipe example', 'julia --project=. examples/reservoir_pipe.jl', timeout=1800)


In [ ]:
run_stage('Droop FCR example', 'julia --project=. examples/fcr_load_step.jl', timeout=1800)


In [ ]:
run_stage('Transient-droop FCR example', 'julia --project=. examples/fcr_transient_droop.jl', timeout=1800)


In [ ]:
run_stage('SMIB transfer-limit disturbance', 'julia --project=. examples/smib/smib_transfer_limit_step.jl', timeout=1800)


In [ ]:
run_stage('Full hydro-grid FCR', 'julia --project=. examples/fcr_full_hydro_grid/full_hydro_grid_fcr.jl', timeout=1800)


In [ ]:
run_stage('Component-wise generator-grid', 'julia --project=. examples/03_generator_infinite_bus/component_smib.jl', timeout=1800)


In [ ]:
run_stage('Full component nonlinear FCR', 'julia --project=. examples/04_full_component_fcr/full_component_fcr.jl', timeout=1800)


In [ ]:
print("\nFINAL RAILWAY VALIDATION SUMMARY")
print("-"*78)
for r in results:
    print(f"{r['status']:4}  {r['stage']:<42} exit={r['exit']}  {r['seconds']:>7.1f}s")
failed=[r for r in results if r["status"]=="FAIL"]
print("-"*78)
print(f"Passed: {len(results)-len(failed)}/{len(results)}")
print(f"Failed: {len(failed)}/{len(results)}")
print("First actionable failure:", failed[0]["stage"] if failed else "None — ready for single-area AGC")
